In [1]:
import requests, re
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font

In [2]:
beamlines = {
    '2-ID':    {'fullname': 'Soft Inelastic X-ray Scattering',
                'tla':      'SIX',
                'program':  'Electronic Structure Techniques',
                'status':   'operating'},
    
    '3-ID':    {'fullname': 'Hard X-ray Nanoprobe',
                'tla':      'HXN',
                'program':  'Imaging and Microscopy',
                'status':   'operating'},
    
    '4-BM':    {'fullname': 'X-ray Fluorescence Microscopy',
                'tla':      'XFM',
                'program':  'BioImaging',
                'status':   'operating'},
    
    '4-ID':    {'fullname': 'Integrated In situ and Resonant Hard X-ray Studies',
                'tla':      'ISR',
                'program':  'Complex Scattering',
                'status':   'operating'},
    
    '5-ID':    {'fullname': 'Submicron Resolution X-ray Spectroscopy',
                'tla':      'SRX',
                'program':  'Imaging and Microscopy',
                'status':   'operating'},
    
    '6-BM':    {'fullname': 'Beamline for Materials Measurement',
                'tla':      'BMM',
                'program':  'Spectroscopy',
                'status':   'operating'},
    
    '6-ID':    {'fullname': 'High Resolution Powder Diffraction',
                'tla':      'HRD',
                'program':  'Hard X-ray Methods',
                'status':   'development'},
    
    '7-BM':    {'fullname': 'Quick x-ray Absorption and Scattering',
                'tla':      'QAS',
                'program':  'Spectroscopy',
                'status':   'operating'},
    
    '7-ID-1':  {'fullname': 'Spectroscopy Soft and Tender',
                'tla':      'SST1',
                'program':  'Spectroscopy',
                'status':   'operating'},
    
    '7-ID-2':  {'fullname': 'Spectroscopy Soft and Tender 2',
                'tla':      'SST2',
                'program':  'Spectroscopy',
                'status':   'operating'},
    
    '8-BM':    {'fullname': 'Tender Energy X-ray Absorption Spectroscopy',
                'tla':      'TES',
                'program':  'Spectroscopy',
                'status':   'operating'},
    
    '8-ID':    {'fullname': 'Inner-Shell Spectroscopy',
                'tla':      'ISS',
                'program':  'Spectroscopy',
                'status':   'operating'},
    
    '9-ID':    {'fullname': 'Coherent Diffraction Imaging',
                'tla':      'CDI',
                'program':  'Imaging and Microscopy',
                'status':   'development'},
    
    '10-ID':   {'fullname': 'Inelastic X-ray Scattering',
                'tla':      'IXS',
                'program':  'Complex Scattering',
                'status':   'operating'},
    
    '11-BM':   {'fullname': 'Complex Materials Scattering',
                'tla':      'CMS',
                'program':  'Complex Scattering',
                'status':   'operating'},
    
    '11-ID':   {'fullname': 'Coherent Hard X-ray Scattering',
                'tla':      'CHX',
                'program':  'Complex Scattering',
                'status':   'operating'},
    
    '12-ID':   {'fullname': 'Soft Matter Interfaces',
                'tla':      'SMI',
                'program':  'Complex Scattering',
                'status':   'operating'},
    
    '13-ID':   {'fullname': 'Advanced Materials Processing',
                'tla':      'AMP',
                'program':  'Complex Scattering',
                'status':   'development'},
    
    '16-BM':   {'fullname': 'Quantitive Cellular Tomography',
                'tla':      'QCT',
                'program':  'Studies Biology',
                'status':   'development'},
    
    '16-ID':   {'fullname': 'Life Science X-ray Scattering',
                'tla':      'LiX',
                'program':  'Structural Biology',
                'status':   'operating'},
    
    '17-BM':   {'fullname': 'X-ray Footprinting of Biological Materials',
                'tla':      'XFP',
                'program':  'Structural Biology',
                'status':   'operating'},
    
    '17-ID-1': {'fullname': 'Highly Automated Macromolecular Crystallography',
                'tla':      'AMX',
                'program':  'Structural Biology',
                'status':   'operating'},
    
    '17-ID-2': {'fullname': 'Frontier Microfocusing Macromolecular Crystallography',
                'tla':      'FMX',
                'program':  'Structural Biology',
                'status':   'operating'},
    
    '18-ID':   {'fullname': 'Full Field X-ray Imaging',
                'tla':      'FXI',
                'program':  'Imaging and Microscopy',
                'status':   'operating'},
    
    '19-ID':   {'fullname': 'Biological Microdiffraction Facility',
                'tla':       'NYX',
                'program':  'Structural Biology',
                'status':   'operating'},
    
    '21-ID':   {'fullname': 'Electron Spectro-Microscopy',
                'tla':      'ESM',
                'program':  'Electronic Structure Techniques',
                'status':   'operating'},
    
    '22-IR-1': {'fullname': 'Frontier Synchrotron Infrared Spectroscopy',
                'tla':      'FIS',
                'program':  'Electronic Structure Techniques',
                'status':   'operating'},
    
    '22-IR-2': {'fullname': 'Magnetospectroscopy, Ellipsometry and Time-Resolved Optical Spectroscopie',
                'tla':      'MET',
                'program':  'Electronic Structure Techniques',
                'status':   'operating'},
    
    '23-ID-1': {'fullname': 'Coherent Soft X-ray Scattering beamline',
                'tla':      'CSX',
                'program':  'Electronic Structure Techniques',
                'status':   'operating'},
    
    '23-ID-2': {'fullname': 'In situ and Operando Soft X-ray Spectroscopy',
                'tla':      'IOS',
                'program':  'Spectroscopy',
                'status':   'operating'},
    
    '24-IR':   {'fullname': 'IR Nanospctroscopy Microspectroscopy',
                'tla':      'INF',
                'program':  'Electronic Structure Techniques',
                'status':   'development'},
    
    '26-ID-1': {'fullname': 'Advanced Nanoscale Imaging',
                'tla':      'ANI',
                'program':  'Imaging and Microscopy',
                'status':   'development'},
    
    '26-ID-1': {'fullname': 'Tender X-ray Nanoprobe',
                'tla':      'TXN',
                'program':  'Imaging and Microscopy',
                'status':   'development'},
    
    '27-ID':   {'fullname': 'High Energy Engineering X-ray Scattering',
                'tla':      'HEX',
                'program':  'Hard X-ray Methods',
                'status':   'operating'},
    
    '28-ID-1': {'fullname': 'Pair Distribution Function',
                'tla':      'PDF',
                'program':  'Hard X-ray Methods',
                'status':   'operating'},
    
    '28-ID-2': {'fullname': 'X-ray Powder Diffraction',
                'tla':      'XPD',
                'program':  'Hard X-ray Methods',
                'status':   'operating'},
    
    '29-ID-1': {'fullname': 'Soft X-ray Nanoprobe',
                'tla':      'SXN',
                'program':  'Imaging and Microscopy',
                'status':   'development'},
    
    '29-ID-2': {'fullname': 'Soft X-ray Photoemission and Scattering Imagine',
                'tla':      'ARI',
                'program':  'Electronic Structure Techniques',
                'status':   'development'},
}

# Fetch beamline publication data from NSLS-II website

In [ ]:
labels = ['port', 'TLA', 'program', 'total', 'high profile', 'years operating', '% high profile', 'pubs per year', 'total citations', 'citations per paper']
frame = pd.DataFrame(columns=labels)

i = -1
for j, port in enumerate(beamlines.keys()):
    if beamlines[port]['status'] == 'development':
        continue
    i = i+1

    ## basic beamline publication data
    tla = beamlines[port]['tla']
    program = beamlines[port]['program']
    print(tla)
    url = f'https://www.bnl.gov/nsls2/beamlines/publications.php?q={port}'
    df_list = pd.read_html(url)
    df = df_list[-1]
    answer = str(df).split('\n')[-1].split()
    years = len(str(df).split('\n')) - 2

    ## figure out citation counts
    html = requests.get(url).content.decode('utf-8')
    matches=re.findall('Cited (\\d+) times', str(html))
    total_citations = sum((int(x) for x in matches))

    frame.loc[len(frame)] = [port, 
                             tla, 
                             program, 
                             int(answer[3]), 
                             int(answer[2]), 
                             years, 
                             round(100*float(answer[2])/float(answer[3])), 
                             round(float(answer[3])/float(years)), 
                             total_citations, 
                             round(total_citations / float(answer[3]))]
    #if j > 3:
    #    break
frame.to_pickle('serilaization')        
frame



# ... or read beamline publication data from serialization

In [15]:
frame = pd.read_pickle('serilaization')
frame.index += 1
#frame

# Display beamline publication results, sorted by total number of publications 

In [16]:
this = frame.sort_values(by='total', ascending=False, ignore_index=True)
this.index +=1
this

,port,TLA,program,total,high profile,years operating,% high profile,pubs per year,total citations,citations per paper
1,7-BM,QAS,Spectroscopy,636,437,9,69,71,31852,50
2,11-BM,CMS,Complex Scattering,623,321,10,52,62,20549,33
3,17-ID-1,AMX,Structural Biology,471,218,13,46,36,9554,20
4,28-ID-2,XPD,Hard X-ray Methods,449,222,13,49,35,21210,47
5,8-ID,ISS,Spectroscopy,358,196,11,55,33,22368,62
6,17-ID-2,FMX,Structural Biology,347,166,13,48,27,7092,20
7,28-ID-1,PDF,Hard X-ray Methods,309,153,10,50,31,6395,21
8,16-ID,LiX,Structural Biology,252,81,13,32,19,3633,14
9,3-ID,HXN,Imaging and Microscopy,246,72,16,29,15,6486,26
10,12-ID,SMI,Complex Scattering,234,98,11,42,21,4079,17


# Display beamline publication results, sorted by number of publications per year

In [17]:
this = frame.sort_values(by='pubs per year', ascending=False, ignore_index=True)
this.index +=1
this

,port,TLA,program,total,high profile,years operating,% high profile,pubs per year,total citations,citations per paper
1,7-BM,QAS,Spectroscopy,636,437,9,69,71,31852,50
2,11-BM,CMS,Complex Scattering,623,321,10,52,62,20549,33
3,17-ID-1,AMX,Structural Biology,471,218,13,46,36,9554,20
4,28-ID-2,XPD,Hard X-ray Methods,449,222,13,49,35,21210,47
5,8-ID,ISS,Spectroscopy,358,196,11,55,33,22368,62
6,28-ID-1,PDF,Hard X-ray Methods,309,153,10,50,31,6395,21
7,17-ID-2,FMX,Structural Biology,347,166,13,48,27,7092,20
8,6-BM,BMM,Spectroscopy,219,79,9,36,24,3987,18
9,12-ID,SMI,Complex Scattering,234,98,11,42,21,4079,17
10,16-ID,LiX,Structural Biology,252,81,13,32,19,3633,14


# Display beamline publication results, sorted by fraction of high impact publications

In [ ]:
this = frame.sort_values(by='% high profile', ascending=False, ignore_index=True)
this.index +=1
this